### Задание
1) Загрузите модель эмбеддингов для русского языка и проанализируйте, какой препроцессинг требуется для успешной работы с ней.  
2) Подготовьте собственный датасет на русском языке, обучите **word2vec** и сравните ближайших соседей с готовыми эмбеддингами.  
3) Постройте индекс по вашему датасету. Если ваши данные — одно длинное произведение, разбейте его на части (абзацы или фрагменты по 100–500 слов). Проверьте работу поиска по вашему индексу на нескольких запросах.

### Что сдавать
1) Список выбранных слов и их ближайшие соседи (готовая русская модель; задание 1).  
2) 5–10 примеров ближайших слов + 2–3 наблюдения (word2vec на ваших данных; задание 2).  
3) Для retrieval: 3–5 запросов и топ-5 документов + короткий вывод (задание 3).


# Домашнее задание

## Импорты

In [1]:
import html
import nltk
import pandas as pd
from collections import Counter
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
russian_stopwords = set(stopwords.words('russian'))
from pymorphy3 import MorphAnalyzer

import re
import numpy as np

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec

## Загрузка данных

Для выполнения задания был выбран датасет, содержащий в себе отзывы из кинопоиска.

In [ ]:
df = pd.read_json("hf://datasets/blinoff/kinopoisk/kinopoisk.jsonl", lines=True)

d:\Shared\Documents\Уник\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df.sample(5)

,part,movie_name,review_id,author,date,title,grade3,grade10,content
18721,top250,Остров проклятых (2009),9891,genry-geil,2010-03-14,NaN,Good,10.0,"\nНе люблю ужастики, которые называются в тако..."
101,top250,Рапунцель Запутанная история (2010),23600,Венцеслава,2010-12-11,А Дисней-то взрослеет…,Good,9.0,"\nФильм (да-да, фильм, а не «мультик»!!!) понр..."
33407,bottom100,Темный мир в 3D (2010),25647,romanist13,2010-12-20,Неожиданно!,Good,8.0,"\n Действительно, этот фильм для меня — о..."
8995,top250,История игрушек Большой побег (2010),24808,Angy_19,2012-01-15,Жизнь игрушек продолжается…,Good,10.0,"\n«История игрушек», несмотря на ее раскрученн..."
25292,top250,Отступники (2006),6312,Claudine,2006-11-09,NaN,Good,0.0,\nМне очень понравился этот фильм: и актеры мо...


Меня интересует столбец content, также можно заметить что текст имеет некотоыре артефакты парсинга, такие как \n. Необходимо будет их удалить.

Имеем следующий пайплайн предобработки - df.content -> список с рецензиями -> удаление html артефактов -> regex очистка текста -> токенизация -> лемматизация -> удаление стоп-слов

In [ ]:
def lemmatize(word, morph, cache):
    if word not in cache:
        cache[word] = morph.parse(word)[0]
    return cache[word]

In [ ]:
OC_TO_UPOS = {
    "NOUN": "NOUN",
    "ADJF": "ADJ",
    "ADJS": "ADJ",
    "COMP": "ADJ",
    "VERB": "VERB",
    "INFN": "VERB",
    "GRND": "VERB",
    "PRTF": "VERB",
    "PRTS": "VERB",
    "ADVB": "ADV",
    "NPRO": "DET",
}

def to_model_format(word, morph, cache):
    if word not in cache:
        p = morph.parse(word)[0]
        lemma = p.normal_form
        lemma = lemma.replace('ё', 'е')
        pos = OC_TO_UPOS.get(p.tag.POS, None)
        cache[word] = f"{lemma}_{pos}" if pos is not None else None
    return cache[word]

content = df.content.values.tolist()
result = []
cache = {}
morph = MorphAnalyzer()

lemmas_russian_stop_words = set(
    w for w in (to_model_format(sw, morph, cache) for sw in russian_stopwords)
    if w is not None
)

for i in range(len(content)):
    content[i] = html.unescape(content[i])
    tokens = re.findall(r"[а-яА-ЯёЁ]+", content[i])
    lemmas = [to_model_format(w, morph, cache) for w in tokens]
    result.append([
        lemma for lemma in lemmas
        if lemma is not None and lemma not in lemmas_russian_stop_words
    ])

Выберем 15 самых популярных слов для теста

In [ ]:
test = [item for sublist in result for item in sublist]
test = Counter(test)
most_common_words = test.most_common(15)
words_to_try = [list(word)[0] for word in most_common_words]

In [ ]:
#Посмотреть, какие модели доступны:
print(list(api.info()['models'].keys())[:30])

MODEL_NAME = "word2vec-ruscorpora-300"  # загружаем модель, обученную на википедии

try:
    wv = api.load(MODEL_NAME)  # KeyedVectors
    print("Загружено:", MODEL_NAME)
    print("Размерность:", wv.vector_size) 
except Exception as e:
    wv = None
    print("Не удалось загрузить готовые эмбеддинги:", repr(e))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']
Загружено: word2vec-ruscorpora-300
Размерность: 300


In [ ]:
def show_neighbors(wv, word, topn=10):
    if wv is None:
        print("Нет загруженных эмбеддингов (wv=None).")
        return

    if word not in wv:
        print(f"'{word}' нет в словаре модели.")
        return

    neigh = wv.most_similar(word, topn=topn)
    print(word, "->", [w for w, _ in neigh])

In [ ]:
for w in words_to_try:
    show_neighbors(wv, w, topn=5)

фильм_NOUN -> ['кинофильм_NOUN', 'кино_NOUN', 'сериал_NOUN', 'телефильм_NOUN', 'кинокартина_NOUN']
'который_ADJ' нет в словаре модели.
человек_NOUN -> ['женщина_NOUN', 'мужчина_NOUN', 'человеческий_ADJ', 'идолопоклонствовать_VERB', 'высокопорядочный_ADJ']
очень_ADV -> ['чрезвычайно_ADV', 'весьма_ADV', 'довольно_ADV', 'необычайно_ADV', 'особенно_ADV']
жизнь_NOUN -> ['жизненный_ADJ', 'бытие_NOUN', 'быт_NOUN', 'существование_NOUN', 'жизнить_VERB']
герой_NOUN -> ['героиня_NOUN', 'персонаж_NOUN', 'глупышкин_NOUN', 'средиземский_NOUN', 'героический_ADJ']
актер_NOUN -> ['артист_NOUN', 'режиссер_NOUN', 'актриса_NOUN', 'актерский_ADJ', 'сценический_ADJ']
'самый_ADJ' нет в словаре модели.
время_NOUN -> ['пора_NOUN', 'ситлуэт_NOUN', 'период_NOUN', 'момент_NOUN', 'промежуток_NOUN']
смотреть_VERB -> ['глядеть_VERB', 'посмотреть_VERB', 'поглядывать_VERB', 'взглядывать_VERB', 'поглядеть_VERB']
каждый_ADJ -> ['воскрес_NOUN', 'прейд_NOUN', 'предтечево_NOUN', 'взяш_NOUN', 'реца_NOUN']
первый_ADJ -> ['вт

актер_NOUN → артист, режиссер, актриса — семантически верно
роль_NOUN → сыграть, играть — модель уловила связь актерской роли с "игрой"
первый_ADJ → порядковые числительные — очень логично
сказать_VERB → глаголы речи — правильно

"Странные" результаты:

каждый_ADJ → какие-то церковнославянские слова 
картина_NOUN → пейзаж, портрет — модель понимает слово в значении живописи, а не фильма.
роль_NOUN → имена актёров — модель обучена на текстах где роли упоминаются вместе с именами

In [ ]:
# Обучение skip-gram: sg=1
# Параметры можно менять
w2v_kinopoisk = Word2Vec(
    sentences=result,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,              # 1 = skip-gram, 0 = CBOW
    negative=10,
    epochs=10,
    workers=4
)

wv_kinopoisk = w2v_kinopoisk.wv
print("Словарь модель, обученной на тексте кинопоиска:", len(wv_kinopoisk))
print("Размерность:", wv_kinopoisk.vector_size)


In [ ]:
movies = ["интерстеллар_NOUN", "джентльмены_NOUN", "1+1_NOUN", "начало_NOUN", "брат_NOUN", "гладиатор_NOUN", "леон_NOUN", "шрэк_NOUN", "матрица_NOUN"]
for m in movies:
    if m in wv_kinopoisk:
        print("\n", m, "->", [w for w, _ in wv_kinopoisk.most_similar(m, topn=10)])
    else:
        print("\n", m, "нет в словаре (проверь min_count или орфографию)")


 интерстеллар_NOUN нет в словаре (проверь min_count или орфографию)

 джентльмены_NOUN нет в словаре (проверь min_count или орфографию)

 1+1_NOUN нет в словаре (проверь min_count или орфографию)

 начало_NOUN -> ['оконцовка_NOUN', 'конец_NOUN', 'середина_NOUN', 'средина_NOUN', 'кинодейство_NOUN', 'притормозить_VERB', 'мутноватый_ADJ', 'обнадежить_VERB', 'сначала_ADV', 'просчитываться_VERB']

 брат_NOUN -> ['старший_NOUN', 'старший_ADJ', 'сестра_NOUN', 'сводный_ADJ', 'младший_ADJ', 'двоюродный_ADJ', 'степа_NOUN', 'отец_NOUN', 'уэйанс_NOUN', 'прабабушка_NOUN']

 гладиатор_NOUN -> ['максимуса_NOUN', 'рассесть_VERB', 'кроу_NOUN', 'ридлить_VERB', 'рассело_NOUN', 'колизей_NOUN', 'гибсон_NOUN', 'центурион_NOUN', 'скотт_NOUN', 'генерал_NOUN']

 леон_NOUN -> ['серджио_NOUN', 'леона_NOUN', 'вестерн_NOUN', 'киллер_NOUN', 'матильда_NOUN', 'серджо_NOUN', 'долларовый_ADJ', 'рено_NOUN', 'иствуд_NOUN', 'клинт_NOUN']

 шрэк_NOUN -> ['шрека_NOUN', 'огре_NOUN', 'рататуй_NOUN', 'огра_NOUN', 'ледниковый_

гладиатор_NOUN → максимуса, кроу, колизей, скотт — модель уловила связь с Расселом Кроу и Ридли Скоттом
леон_NOUN → матильда, рено, киллер, вестерн — точно поймал контекст фильма
шрэк_NOUN → рататуй, мультфильм, суперсемейка — анимационный кластер
матрица_NOUN → вачовск, аватар, виртуальный — sci-fi кластер

Сравним с word2vec-ruscorpora-300.

In [ ]:
movies = ["интерстеллар_NOUN", "джентльмены_NOUN", "1+1_NOUN", "начало_NOUN", "брат_NOUN", "гладиатор_NOUN", "леон_NOUN", "шрэк_NOUN", "матрица_NOUN"]

for m in movies:
    show_neighbors(wv, m, topn=15)

'интерстеллар_NOUN' нет в словаре модели.
'джентльмены_NOUN' нет в словаре модели.
'1+1_NOUN' нет в словаре модели.
начало_NOUN -> ['конец_NOUN', 'середина_NOUN', 'korea::неrald_NOUN', 'сябринец_NOUN', 'разгар_NOUN', 'начинаться_VERB', 'прошлый_ADJ', 'девятидесятый_ADJ', 'ситлуэт_NOUN', 'хix_NOUN', 'восьмидесятый_ADJ', 'период_NOUN', 'завершение_NOUN', 'тридцатый_ADJ', 'семидесятый_ADJ']
брат_NOUN -> ['двоюродный_ADJ', 'сестра_NOUN', 'племянник_NOUN', 'сын_NOUN', 'отец_NOUN', 'дядя_NOUN', 'зять_NOUN', 'племянница_NOUN', 'атя_NOUN', 'младший_ADJ', 'эмухварь_NOUN', 'братец_NOUN', 'бабенчиковый_ADJ', 'шурин_NOUN', 'сестренич_NOUN']
гладиатор_NOUN -> ['гладиаторский_ADJ', 'немейский_ADJ', 'salutant_NOUN', 'morituri::te_NOUN', 'мирмиллон_NOUN', 'ретиарий_NOUN', 'атлет_NOUN', 'арена_NOUN', 'цезарь_NOUN', 'патрокл_NOUN', 'акретон_NOUN', 'древнеримский_ADJ', 'цирк_NOUN', 'колизей_NOUN', 'махотина_NOUN']
леон_NOUN -> ['жан_NOUN', 'меес_NOUN', 'абгарович::орбеть_VERB', 'реймон_NOUN', 'валерио_NO

In [ ]:
wv_kinopoisk['блеф_NOUN']

array([-0.26840374, -0.16733882,  0.20627765, -0.56218225,  0.28764975,
       -0.58307993,  0.10852042,  0.2837838 ,  0.04698284, -0.5431272 ,
       -0.05835436, -0.32221463, -0.03958735,  0.6001261 , -0.38028997,
        0.30193436, -0.06036001,  0.39917353, -0.6758961 , -0.28265423,
        0.10254414,  0.917851  ,  0.0987336 ,  0.27227962, -0.05692919,
       -0.00280387,  0.31115887, -0.03673532, -0.01736858, -0.39389235,
        0.048564  ,  0.32048368, -0.15799499,  0.2290145 , -0.07746954,
        0.304511  , -0.30029067, -0.49356693,  0.9382713 ,  0.08790235,
       -0.8032912 ,  0.29366225, -0.40059447, -0.14718461, -0.390379  ,
        0.37244567, -0.54264134,  0.38335437,  0.83864814,  0.41704383,
        0.3207309 ,  0.6345861 ,  0.34607586,  0.13414101, -0.16286775,
       -0.19313335,  0.088763  , -0.36795145, -0.333823  , -0.25109416,
       -0.4179169 ,  0.74067175, -0.59172535,  0.62238085, -0.36079803,
       -0.40032643,  0.5180684 ,  0.05890998, -0.22415943,  0.53

In [ ]:
def doc_vector(tokens, wv):
    # Шаг 1 - фильтруем токены. Если есть в словаре нашей модели - берем их.
    tokens_filtered = [token for token in tokens if token in wv]
    if tokens_filtered == []:
        return np.zeros(wv.vector_size)
    # Шаг 2 - Для каждого оставшегося токена берем его вектор из нашей модели - это массив размером 100
    token_vec_list = [wv[token] for token in tokens_filtered]
    # Шаг 3 - Усредняем получившиеся массивы
    means = np.mean(token_vec_list, axis=0)
    
    return means

In [ ]:
# Применим написанную функцию ко всем токенам нашего документа
D_matrix = [doc_vector(token, wv_kinopoisk) for token in result]

In [ ]:
# Проверим размерность нашего результата
np.array(D_matrix).shape

(36591, 100)

Теперь необходимо нормировать строки. Реализуем функцию для этого.

In [ ]:
def normalize_rows(matrix):    
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)

In [ ]:
# Применим написанную функцию к изначальной матрице
D_matrix_normalized = normalize_rows(D_matrix)

C:\Users\bogd\AppData\Local\Temp\ipykernel_23044\1980124519.py:2: RuntimeWarning: invalid value encountered in divide
  return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)


In [ ]:
# Проверим размерность получившейся матрицы
D_matrix_normalized.shape

(36591, 100)

Отлично, размеры верные, можно приступать к написанию функции поиска.

Workflow алгоритма следующий: 
1. Токенизировать запрос
2. Применить doc_vector -> получить векторы запроса
3. Нормализовать векторы запроса
4. Посчитать cosine similarity D_norm @ q_norm
5. Взять топ-k индексов с наибольшим score

In [ ]:
def search(query, D_norm, wv, k=5):
    # На вход мы получаем запрос, который необходимо токенизировать в соотвествии с нашей моделью, функция для этого уже реализована выше


    # ШАГ 1 - ТОКЕНИЗИРУЕМ ЗАПРОС
    query_tokens = []
    cache = {}

    tokens = re.findall(r"[а-яА-ЯёЁ]+", html.unescape(query))
    lemmas = [l for l in (to_model_format(w, morph, cache) for w in tokens) if l is not None and l not in lemmas_russian_stop_words]
        
    # ШАГ 2 - применяем функцию для векторизации нашего запроса
    query_matrix = doc_vector(lemmas, wv)

    # ШАГ 3 - нормализуем вектор
    query_matrix_norm = query_matrix / np.linalg.norm(query_matrix)

    # ШАГ 4 - посчитаем cosine similarity D_norm @ q_norm
    cosine_similarity = D_norm @ query_matrix_norm

    # ШАГ 5 - берем топ индексов с наибольшим score
    result = np.argsort(-cosine_similarity)[:k]

    return result    
    

In [ ]:
indices = search("великолепная игра актёров", D_matrix_normalized, wv_kinopoisk)
for idx in indices:
    print(f"Индекс: {idx}")
    print(df.iloc[idx]['content'][:300])

Индекс: 7696

Фильм отличный. Актеры сыграли великолепно, особенно хотелось бы отметить игру Кевина Спейси. Очень, очень разноплановый актер. По моему он может сыграть любой образ.

Индекс: 1474

Просто бесподобный фильм! Отличная режессура, великолепная актерская игра, идея, атмосфера. Здесь есть все, и все на высшем уровне. Мэлу огромный респект. Его работы ни на что не похожи.

10 из 10.
Индекс: 17386

Фильм конено потрясающий, игра актеров тоже, кроме, как справедливо заметили, Кетчера. Был бы кто-нибудь другой на его месте (актер поинтересней) фильм был бы ещё успешней.
Индекс: 10314

Согласен абсолютно! Фильм шикарный особенно хочется отметить блистательный актерский состав! Диалог сэра Бена Кингсли и Моргана Фримана это просто гениально!
Индекс: 6788

Первоклассный триллер с отличным актёрским составом. Кроме уже упомянутых актёров хочу выделить также весьма талантливого, но не особо популярного Питера Грина, исполнившего небольшую роль Ретфута. 

Также безумно понравилась велик

Отлично! запросы работают! Попробуем еще парочку запросов.

In [ ]:
indices = search("страшный фильм напугал", D_matrix_normalized, wv_kinopoisk)
for idx in indices:
    print(f"Индекс: {idx}")
    print(df.iloc[idx]['content'][:300])

Индекс: 28680

В этом черно-белом фильме нет маньяков и истеричных криков блондинок, которых душит психбольной. Тут страшнее. Страшно от того, что это было на самом деле. Страшно, ведь это история. И только в наших силах не повторить ошибок времени. 

Этот фильм должен посмотреть каждый. В нем нет ничего нового, 
Индекс: 32617

      Помню рекламу к этому фильму: практически «звонок» в русском варианте! Я люблю ужасы, но это — «кошмар»! В свое время додумалась сходить на этот фильм в кино. Моя самая большая ошибка.



Идея фильма интересная. Мстительные призраки маленьких девочек — это пугает и заставляет холод проникать 
Индекс: 29064

Фильм смотрела раз двадцать, но каждый раз боюсь, как в первый раз! Одна просто не могу смотреть, хотя первый раз смотрела одна. Этот фильм, по-моему, лучший вариант мистического триллера, в нем нет сцен насилия, убийства, но то напряжение, которое наступает с крика на первой минуте, не отпускает д
Индекс: 11727

Фильм начал мне нравиться только примерн

In [ ]:
indices = search("лучший фильм всех времен и народов", D_matrix_normalized, wv_kinopoisk)
for idx in indices:
    print(f"Индекс: {idx}")
    print(df.iloc[idx]['content'][:300])

Индекс: 28690

Этот фильм — шок. Один из самых великих фильмов в истории. Такие фильмы нужны для того, чтобы встряхивать людей время от времени и напоминать что такое фашизм. А то что-то в последнее время начали забывать.
Индекс: 13373

Вот смотришь русские фильмы времён СССР и понимаешь, цензура это не так уж и плохо. Столько добрых, светлых фильмов снято в это время. Пусть нет спецэффектов, пусть с высоты 2009 года некоторые сцены и смотрятся наивно, пусть.

Этот фильм — гордость нации, достояние русского народа наряду с великим
Индекс: 1565

Лучший фильм всех времен и народов! Шедевр мирового кино! Создатели фильма просто киногении! Ни до, ни после ничего лучше не сняли, хотя мне бы хотелось посмотреть продолжение, эдак 10 лет спустя…
Индекс: 27614

Общепризнанный эталон боевика. Лучший фантастичестий боевик всех времён и народов. Фильм который будет со мной всегда. Фильм который можно смотреть снова и снова. Фильм ставший легендой прошедшего века…
Индекс: 24479

Этот фильм — образе